# 📋 Nettoyage complet des données - Commandes

Ce notebook gère le nettoyage et le filtrage métier des commandes (`olist_orders_dataset.csv`).


## 1. Configuration des chemins et du fichier

Initialisation du répertoire.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import os

RAW_DIR = Path(r"C:\Users\malek\OneDrive\Desktop\ESPRIT\3ème IA\PROJET BI\data\raw")
PROCESSED_DIR = RAW_DIR.parent / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FILE_NAME = "olist_orders_dataset.csv"
input_path = RAW_DIR / FILE_NAME
output_path = PROCESSED_DIR / f"{input_path.stem}_cleaned.csv"


## 2. Vérification et lecture du fichier

Lecture initiale du fichier des commandes.


In [ ]:
if not input_path.exists():
    raise FileNotFoundError(f"Fichier introuvable : {input_path}")

df = pd.read_csv(input_path, encoding="utf-8-sig")

print("Dimensions initiales :", df.shape)
print("\nColonnes initiales :")
print(df.columns.tolist())
print("\nTypes initiaux :")
print(df.dtypes)


## 3. Nettoyage des noms de colonnes

Normalisation des noms de colonnes.


In [ ]:
df.columns = df.columns.str.replace(r"[^\w]", "", regex=True)

print("\nColonnes après nettoyage des noms :")
print(df.columns.tolist())


## 4. Nettoyage des colonnes textuelles sélectionnées

Soin apporté aux chaînes de caractères `customer_id`, `order_id` et `order_status`.


In [ ]:
colonnes_texte = [
    "customer_id",
    "order_id",
    "order_status",
]

for col in colonnes_texte:
    df[col] = df[col].str.strip()
    df[col] = df[col].replace("", pd.NA)
    df[col] = df[col].str.replace(r"[^\w]", "", regex=True)


## 5. Nettoyage et conversion des dates

Conversion en `datetime` de toutes les étapes temporelles clés.


In [ ]:
colonnes_dates = [
    "order_approved_at",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in colonnes_dates:
    df[col] = df[col].str.strip()
    df[col] = df[col].replace("", pd.NA)
    df[col] = pd.to_datetime(df[col], errors="coerce")


## 6. Suppression des colonnes non conservées

Retrait des horodatages non requis (`order_purchase_timestamp`, `order_delivered_carrier_date`).


In [ ]:
columns_to_remove = [
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
]

df = df.drop(columns=columns_to_remove)

print("\nColonnes restantes :")
print(df.columns.tolist())


## 7. Suppression des lignes/colonnes vides et des doublons

Vérification des lignes vides et suppression des doublons.


In [ ]:
df = df.dropna(axis=0, how="all")
df = df.dropna(axis=1, how="all")

duplicates = df.duplicated().sum()
df = df.drop_duplicates()


## 8. Vérification des plages de dates

Affichage des intervalles temporels enregistrés pour déceler d'éventuelles anomalies.


In [ ]:
print("\nDate minimale d'approbation :", df["order_approved_at"].min())
print("Date maximale d'approbation :", df["order_approved_at"].max())
print("\nDate minimale de livraison client :", df["order_delivered_customer_date"].min())
print("Date maximale de livraison client :", df["order_delivered_customer_date"].max())
print("\nDate minimale de livraison estimée :", df["order_estimated_delivery_date"].min())
print("Date maximale de livraison estimée :", df["order_estimated_delivery_date"].max())


## 9. Filtrage des commandes conservées

Règle métier fondamentale : Conservation exclusive des commandes `delivered` avec dates d'approbation et de livraison valides.


In [ ]:
print("\nRépartition des statuts avant filtrage :")
print(df["order_status"].value_counts())

df = df[df["order_status"] == "delivered"].copy()
df = df.dropna(subset=["order_approved_at"]).copy()
df = df.dropna(subset=["order_delivered_customer_date"]).copy()

print("\nRépartition des statuts après filtrage :")
print(df["order_status"].value_counts())


## 10. Bilan final du nettoyage

Récapitulatif de la table nettoyée.


In [ ]:
print("\nTypes finaux :")
print(df.dtypes)
print("\nNettoyage terminé.")
print("Doublons supprimés :", duplicates)
print("\nValeurs manquantes par colonne :")
print(df.isna().sum())
print("\nDimensions finales :", df.shape)


## 11. Export du fichier nettoyé

Exportation vers le dossier `processed`.


In [ ]:
df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
    na_rep="NULL",
)

print("\nCopie enregistrée :", output_path)
print("Le fichier original reste intact.")
